# cMSCI — Full GPU Pipeline

**All 4 GPU tasks in one notebook:**

| Task | Description | Time Est. |
|------|------------|----------|
| 1 | Download AudioCaps (1000 samples) + expand training triples | ~15 min |
| 2 | Retrain bridge + prob adapters on expanded data | ~10 min |
| 3 | Re-optimize cMSCI hyperparameters (windowed CLAP) | ~5 min |
| 4 | Scale benchmark evaluation to 1000 samples | ~15 min |
| 5 | (Optional) VLM-as-Judge if Ollama available | ~30 min |

## Setup

Upload the **entire project** to your GPU environment, or clone it:
```bash
git clone <your-repo-url> MultiModal-Coherence-AI
cd MultiModal-Coherence-AI
pip install -r requirements.txt
```

Then open this notebook from inside the project directory.

**Expected total time: ~45-60 min on A6000 / ~90 min on T4**

## Cell 0: Setup & GPU Check

In [ ]:
import os
import sys
import time
import json
import subprocess
from pathlib import Path

import numpy as np
import torch

# Find project root
NOTEBOOK_DIR = Path(os.getcwd())
if (NOTEBOOK_DIR / "src").exists():
    PROJECT_ROOT = NOTEBOOK_DIR
elif (NOTEBOOK_DIR.parent / "src").exists():
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    # Try common locations
    for candidate in [Path.home() / "MultiModal-Coherence-AI",
                      Path("/workspace/MultiModal-Coherence-AI"),
                      Path("/content/MultiModal-Coherence-AI")]:
        if (candidate / "src").exists():
            PROJECT_ROOT = candidate
            break
    else:
        raise RuntimeError("Cannot find project root. Run this notebook from the project directory.")

os.chdir(str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT))

print("=" * 60)
print("PROJECT SETUP")
print("=" * 60)
print(f"  Project root: {PROJECT_ROOT}")
print(f"  Working dir:  {os.getcwd()}")

# GPU check
if torch.cuda.is_available():
    DEVICE = "cuda"
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  GPU: {gpu_name} ({gpu_mem:.1f} GB)")
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    DEVICE = "mps"
    print("  Device: Apple MPS")
else:
    DEVICE = "cpu"
    print("  WARNING: No GPU found!")

print(f"  PyTorch: {torch.__version__}")
print(f"  Device: {DEVICE}")

# Verify key files exist
required = [
    "src/config/settings.py",
    "src/coherence/cmsci_engine.py",
    "src/embeddings/aligned_embeddings.py",
    "artifacts/cmsci_calibration.json",
    "runs/rq3/rq3_samples.json",
]
missing = [f for f in required if not (PROJECT_ROOT / f).exists()]
if missing:
    print(f"\n  MISSING FILES: {missing}")
    print("  Make sure you uploaded the full project!")
else:
    print("  All required files present!")
print("=" * 60)

## Cell 0b: Install Dependencies (run once)

In [ ]:
# Uncomment and run if dependencies are missing
# !pip install -q transformers datasets soundfile librosa scipy scikit-learn matplotlib
# !pip install -q torchcodec  # needed for HuggingFace audio datasets

---
## Task 1: Download AudioCaps (1000 samples) + Expand Training Triples

Downloads 1000 AudioCaps samples from HuggingFace (`OpenSound/AudioCaps`) using streaming mode (only downloads what we need). Then embeds captions via CLIP text encoder and audio via CLAP to create new training pairs for the bridge.

In [ ]:
print("=" * 60)
print("TASK 1a: Download AudioCaps (1000 samples)")
print("=" * 60)

t0 = time.time()
!python scripts/download_benchmarks.py --audiocaps --huggingface --max-samples 1000
print(f"\nDownload completed in {time.time()-t0:.0f}s")

# Verify manifest
manifest_path = PROJECT_ROOT / "data" / "benchmarks" / "audiocaps" / "manifest.json"
if manifest_path.exists():
    with open(manifest_path) as f:
        m = json.load(f)
    print(f"Manifest: {len(m.get('entries', []))} entries")
else:
    print("ERROR: Manifest not found!")

In [ ]:
print("=" * 60)
print("TASK 1b: Embed AudioCaps triples + Augment")
print("=" * 60)

t0 = time.time()
!python scripts/prepare_bridge_data.py --audiocaps-triples --augment --augment-factor 3 --report-stats
print(f"\nTriple preparation completed in {time.time()-t0:.0f}s")

# Check output
for npz_name in ["audiocaps_triples.npz", "combined_training_v3.npz", "combined_training.npz"]:
    p = PROJECT_ROOT / "data" / "bridge_training" / npz_name
    if p.exists():
        data = np.load(p)
        keys = list(data.keys())
        sizes = {k: data[k].shape for k in keys if hasattr(data[k], 'shape')}
        print(f"  {npz_name}: {sizes}")

---
## Task 2: Retrain Bridge + Probabilistic Adapters on Expanded Data

Retrains the CrossSpaceBridge and ProbVLM adapters using the expanded training data (original 2,193 pairs + AudioCaps triples + augmentation).

In [ ]:
print("=" * 60)
print("TASK 2a: Load Expanded Training Data")
print("=" * 60)

# Try expanded data first, fall back to original
data_path = None
for candidate in [
    PROJECT_ROOT / "data" / "bridge_training" / "combined_training_v3.npz",
    PROJECT_ROOT / "data" / "bridge_training" / "combined_training.npz",
]:
    if candidate.exists():
        data_path = candidate
        break

if data_path is None:
    raise FileNotFoundError("No training data found! Run Task 1 first.")

data = np.load(data_path)

# Handle different key names
if "image_embeddings" in data:
    image_embs = data["image_embeddings"]
    audio_embs = data["audio_embeddings"]
elif "text_clip_embeddings" in data:
    # AudioCaps triples use text_clip as proxy for image
    image_embs = data["text_clip_embeddings"]
    audio_embs = data["audio_embeddings"]
else:
    keys = list(data.keys())
    raise KeyError(f"Unexpected keys in {data_path}: {keys}")

print(f"  Loaded: {data_path.name}")
print(f"  Image/text embeddings: {image_embs.shape}")
print(f"  Audio embeddings:      {audio_embs.shape}")
print(f"  Total pairs: {len(image_embs)}")

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, random_split

# === CrossSpaceBridge ===

class BridgeProjectionHead(nn.Module):
    def __init__(self, input_dim=512, hidden_dim=384, output_dim=256, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim, bias=True),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, output_dim, bias=False),
        )
    def forward(self, x):
        return F.normalize(self.net(x), p=2, dim=-1)

class CrossSpaceBridge(nn.Module):
    def __init__(self, clip_dim=512, clap_dim=512, bridge_dim=256, hidden_dim=384, dropout=0.1):
        super().__init__()
        self.image_proj = BridgeProjectionHead(clip_dim, hidden_dim, bridge_dim, dropout)
        self.audio_proj = BridgeProjectionHead(clap_dim, hidden_dim, bridge_dim, dropout)
        self.config = dict(clip_image_dim=clip_dim, clap_audio_dim=clap_dim,
                           bridge_dim=bridge_dim, hidden_dim=hidden_dim, dropout=dropout)
    def forward(self, image_emb=None, audio_emb=None):
        result = {}
        if image_emb is not None: result["image"] = self.image_proj(image_emb)
        if audio_emb is not None: result["audio"] = self.audio_proj(audio_emb)
        return result
    def compute_similarity(self, image_emb_np, audio_emb_np):
        self.eval()
        with torch.no_grad():
            img = torch.tensor(image_emb_np, dtype=torch.float32).unsqueeze(0)
            aud = torch.tensor(audio_emb_np, dtype=torch.float32).unsqueeze(0)
            proj = self.forward(image_emb=img, audio_emb=aud)
            return float(F.cosine_similarity(proj["image"], proj["audio"]).item())
    def save(self, path):
        p = Path(path); p.parent.mkdir(parents=True, exist_ok=True)
        torch.save(self.state_dict(), p)
        with p.with_suffix('.json').open('w') as f: json.dump(self.config, f, indent=2)
    @classmethod
    def load(cls, path):
        p = Path(path)
        with p.with_suffix('.json').open('r') as f: config = json.load(f)
        model = cls(**config)
        model.load_state_dict(torch.load(p, map_location='cpu', weights_only=True))
        return model.eval()

# === ProbabilisticAdapter ===

class ProbabilisticAdapter(nn.Module):
    def __init__(self, input_dim=512, hidden_dim=256, num_layers=3, dropout=0.1):
        super().__init__()
        self.input_dim = input_dim
        layers = []
        in_d = input_dim
        for _ in range(num_layers - 1):
            layers.extend([nn.Linear(in_d, hidden_dim), nn.ReLU(), nn.Dropout(dropout)])
            in_d = hidden_dim
        self.backbone = nn.Sequential(*layers)
        self.mu_head = nn.Linear(hidden_dim, input_dim)
        self.alpha_head = nn.Linear(hidden_dim, input_dim)
        self.beta_head = nn.Linear(hidden_dim, input_dim)
        self.config = dict(input_dim=input_dim, hidden_dim=hidden_dim,
                           num_layers=num_layers, dropout=dropout)
    def forward(self, embedding):
        h = self.backbone(embedding)
        mu = embedding + self.mu_head(h)
        alpha = F.softplus(self.alpha_head(h)) + 1e-6
        beta = F.softplus(self.beta_head(h)) + 1e-6
        return mu, alpha, beta
    def uncertainty(self, embedding_np):
        self.eval()
        emb = embedding_np.squeeze()
        if emb.ndim == 1: emb = emb[np.newaxis, :]
        with torch.no_grad():
            _, alpha, _ = self.forward(torch.tensor(emb, dtype=torch.float32))
            return float(alpha.mean().item())
    def save(self, path):
        p = Path(path); p.parent.mkdir(parents=True, exist_ok=True)
        torch.save(self.state_dict(), p)
        with p.with_suffix('.json').open('w') as f: json.dump(self.config, f, indent=2)
    @classmethod
    def load(cls, path):
        p = Path(path)
        with p.with_suffix('.json').open('r') as f: config = json.load(f)
        model = cls(**config)
        model.load_state_dict(torch.load(p, map_location='cpu', weights_only=True))
        return model.eval()

# === Datasets & Losses ===

class ImageAudioPairDataset(Dataset):
    def __init__(self, image_embeddings, audio_embeddings):
        self.images = torch.tensor(image_embeddings, dtype=torch.float32)
        self.audio = torch.tensor(audio_embeddings, dtype=torch.float32)
    def __len__(self): return len(self.images)
    def __getitem__(self, idx): return {"image": self.images[idx], "audio": self.audio[idx]}

class EmbeddingPairDataset(Dataset):
    def __init__(self, inputs, targets):
        self.inputs = torch.tensor(inputs, dtype=torch.float32)
        self.targets = torch.tensor(targets, dtype=torch.float32)
    def __len__(self): return len(self.inputs)
    def __getitem__(self, idx): return self.inputs[idx], self.targets[idx]

class BridgeInfoNCELoss(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.log_temperature = nn.Parameter(torch.tensor(np.log(1.0 / temperature)))
    @property
    def temperature(self): return torch.exp(-self.log_temperature)
    def forward(self, image_emb, audio_emb):
        batch_size = image_emb.size(0)
        logits = torch.mm(image_emb, audio_emb.t()) / self.temperature
        labels = torch.arange(batch_size, device=logits.device)
        loss = 0.5 * (F.cross_entropy(logits, labels) + F.cross_entropy(logits.t(), labels))
        with torch.no_grad():
            acc_i2a = (logits.argmax(1) == labels).float().mean()
            acc_a2i = (logits.t().argmax(1) == labels).float().mean()
        return loss, {"loss": loss.item(), "acc_i2a": acc_i2a.item(),
                      "acc_a2i": acc_a2i.item(), "temp": self.temperature.item()}

class GenGaussNLL(nn.Module):
    def forward(self, mu, alpha, beta, target):
        residual = torch.abs(target - mu)
        alpha_c = torch.clamp(alpha, min=1e-6)
        return (torch.log(alpha_c) + (residual / alpha_c).pow(beta)).mean()

print("Models and training components defined!")

In [ ]:
print("=" * 60)
print("TASK 2b: Train CrossSpaceBridge")
print("=" * 60)

BRIDGE_DIR = PROJECT_ROOT / "models" / "bridge"
BRIDGE_DIR.mkdir(parents=True, exist_ok=True)

BRIDGE_EPOCHS = 50
BRIDGE_BATCH_SIZE = 64
BRIDGE_LR = 3e-4
BRIDGE_PATIENCE = 10

full_dataset = ImageAudioPairDataset(image_embs, audio_embs)
n_val = max(1, int(len(full_dataset) * 0.15))
n_train = len(full_dataset) - n_val
train_data, val_data = random_split(
    full_dataset, [n_train, n_val],
    generator=torch.Generator().manual_seed(42)
)
train_loader = DataLoader(train_data, batch_size=BRIDGE_BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_data, batch_size=BRIDGE_BATCH_SIZE, shuffle=False)

bridge = CrossSpaceBridge().to(DEVICE)
loss_fn = BridgeInfoNCELoss().to(DEVICE)
optimizer = torch.optim.AdamW(
    list(bridge.parameters()) + list(loss_fn.parameters()),
    lr=BRIDGE_LR, weight_decay=1e-4
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=BRIDGE_EPOCHS)

n_params = sum(p.numel() for p in bridge.parameters())
print(f"  Train: {n_train}, Val: {n_val}, Params: {n_params:,}")
print(f"  Device: {DEVICE}")
print()

best_val_loss = float("inf")
patience_counter = 0
t_start = time.time()

for epoch in range(BRIDGE_EPOCHS):
    bridge.train(); loss_fn.train()
    epoch_metrics = []
    for batch in train_loader:
        img = batch["image"].to(DEVICE)
        aud = batch["audio"].to(DEVICE)
        optimizer.zero_grad()
        proj = bridge(image_emb=img, audio_emb=aud)
        loss, metrics = loss_fn(proj["image"], proj["audio"])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(bridge.parameters(), max_norm=1.0)
        optimizer.step()
        epoch_metrics.append(metrics)
    scheduler.step()
    avg = {k: np.mean([m[k] for m in epoch_metrics]) for k in epoch_metrics[0]}

    bridge.eval(); loss_fn.eval()
    val_losses = []
    with torch.no_grad():
        for batch in val_loader:
            proj = bridge(image_emb=batch["image"].to(DEVICE), audio_emb=batch["audio"].to(DEVICE))
            vloss, _ = loss_fn(proj["image"], proj["audio"])
            val_losses.append(vloss.item())
    val_loss = np.mean(val_losses)

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"  Epoch {epoch+1:3d}/{BRIDGE_EPOCHS}: loss={avg['loss']:.4f} "
              f"acc_i2a={avg['acc_i2a']:.3f} acc_a2i={avg['acc_a2i']:.3f} "
              f"val_loss={val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        bridge.save(BRIDGE_DIR / "bridge_best.pt")
    else:
        patience_counter += 1
        if patience_counter >= BRIDGE_PATIENCE:
            print(f"  Early stopping at epoch {epoch+1}")
            break

bridge.save(BRIDGE_DIR / "bridge_final.pt")
print(f"\n  Bridge training complete in {time.time()-t_start:.1f}s")
print(f"  Best val_loss: {best_val_loss:.4f}")
print(f"  Saved: {BRIDGE_DIR / 'bridge_best.pt'}")

In [ ]:
print("=" * 60)
print("TASK 2c: Train Probabilistic Adapters")
print("=" * 60)

ADAPTER_DIR = PROJECT_ROOT / "models" / "prob_adapters"
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

ADAPTER_EPOCHS = 100
ADAPTER_BATCH_SIZE = 32
ADAPTER_LR = 1e-4
ADAPTER_PATIENCE = 15


def train_prob_adapter(embeddings, name, output_path):
    print(f"\n--- Training {name} Adapter ---")
    rng = np.random.default_rng(42)
    targets = embeddings + rng.normal(0, 0.01, size=embeddings.shape).astype(np.float32)
    dataset = EmbeddingPairDataset(embeddings, targets)
    n_val = max(1, int(len(dataset) * 0.15))
    n_train = len(dataset) - n_val
    train_ds, val_ds = random_split(dataset, [n_train, n_val],
                                     generator=torch.Generator().manual_seed(42))
    train_loader = DataLoader(train_ds, batch_size=ADAPTER_BATCH_SIZE, shuffle=True,
                              drop_last=len(train_ds) > ADAPTER_BATCH_SIZE)
    val_loader = DataLoader(val_ds, batch_size=ADAPTER_BATCH_SIZE, shuffle=False)

    adapter = ProbabilisticAdapter(input_dim=512).to(DEVICE)
    optimizer = torch.optim.AdamW(adapter.parameters(), lr=ADAPTER_LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=ADAPTER_EPOCHS)
    l1_loss = nn.L1Loss()
    gg_loss = GenGaussNLL()

    n_params = sum(p.numel() for p in adapter.parameters())
    print(f"  Train: {n_train}, Val: {n_val}, Params: {n_params:,}")

    best_val_loss = float("inf")
    patience_counter = 0
    t_start = time.time()

    for epoch in range(ADAPTER_EPOCHS):
        adapter.train()
        train_losses = []
        for inp, tgt in train_loader:
            inp, tgt = inp.to(DEVICE), tgt.to(DEVICE)
            optimizer.zero_grad()
            mu, alpha, beta = adapter(inp)
            loss = l1_loss(mu, tgt) + gg_loss(mu, alpha, beta, tgt)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(adapter.parameters(), max_norm=1.0)
            optimizer.step()
            train_losses.append(loss.item())
        scheduler.step()

        adapter.eval()
        val_losses = []
        with torch.no_grad():
            for inp, tgt in val_loader:
                mu, alpha, beta = adapter(inp.to(DEVICE))
                loss = l1_loss(mu, tgt.to(DEVICE)) + gg_loss(mu, alpha, beta, tgt.to(DEVICE))
                val_losses.append(loss.item())

        avg_val = np.mean(val_losses)
        if (epoch + 1) % 20 == 0 or epoch == 0:
            print(f"  Epoch {epoch+1:3d}/{ADAPTER_EPOCHS}: train={np.mean(train_losses):.4f}  val={avg_val:.4f}")

        if avg_val < best_val_loss:
            best_val_loss = avg_val
            patience_counter = 0
            adapter.save(output_path)
        else:
            patience_counter += 1
            if patience_counter >= ADAPTER_PATIENCE:
                print(f"  Early stopping at epoch {epoch+1}")
                break

    print(f"  {name} done in {time.time()-t_start:.1f}s (best val={best_val_loss:.4f})")
    return ProbabilisticAdapter.load(output_path)


# Train CLIP adapter (on image/text-proxy embeddings)
clip_adapter = train_prob_adapter(
    image_embs, "CLIP", str(ADAPTER_DIR / "clip_adapter.pt")
)

# Train CLAP adapter (on audio embeddings)
clap_adapter = train_prob_adapter(
    audio_embs, "CLAP", str(ADAPTER_DIR / "clap_adapter.pt")
)

print("\n  Both adapters trained and saved!")

---
## Task 3: Re-optimize cMSCI Hyperparameters

With windowed CLAP now active, the old hyperparameters (alpha=16, w_ti=0.90, etc.) were tuned on single-clip embeddings. Re-optimizing on the new embeddings should recover the correlation from 0.481 back toward 0.60+.

This runs a full grid search over 86K+ configurations with LOO-CV.

In [ ]:
print("=" * 60)
print("TASK 3a: Clear embedding cache (force re-embed with windowed CLAP)")
print("=" * 60)

import shutil

cache_dir = PROJECT_ROOT / ".cache" / "embeddings"
if cache_dir.exists():
    n_files = len(list(cache_dir.rglob("*")))
    shutil.rmtree(cache_dir)
    print(f"  Cleared {n_files} cached files")
else:
    print("  No cache to clear")

# Rebuild embedding indexes
print("\nRebuilding embedding indexes...")
t0 = time.time()
!python scripts/build_embedding_indexes.py
print(f"  Done in {time.time()-t0:.0f}s")

In [ ]:
print("=" * 60)
print("TASK 3b: Re-optimize cMSCI (full grid + LOO-CV)")
print("=" * 60)

t0 = time.time()
!python scripts/optimize_cmsci.py --top 10
print(f"\nOptimization completed in {time.time()-t0:.0f}s")

In [ ]:
print("=" * 60)
print("TASK 3c: Review optimized parameters")
print("=" * 60)

# The optimizer outputs results but doesn't modify settings.py
# You need to manually update src/config/settings.py with the best params

print()
print("IMPORTANT: Check the optimization output above.")
print("If the new best config differs from current settings, update them below:")
print()
print("Current settings (src/config/settings.py):")
from src.config import settings
print(f"  CMSCI_MARGIN_ALPHA    = {settings.CMSCI_MARGIN_ALPHA}")
print(f"  CMSCI_CHANNEL_WEIGHT_TI = {settings.CMSCI_CHANNEL_WEIGHT_TI}")
print(f"  CMSCI_CALIBRATION_MODE  = {settings.CMSCI_CALIBRATION_MODE}")
print(f"  CMSCI_W_3D            = {settings.CMSCI_W_3D}")
print(f"  CMSCI_GAMMA           = {settings.CMSCI_GAMMA}")
print()
print("If the optimizer found better params, update settings.py and re-run cells below.")
print("The optimizer is READ-ONLY — it does not modify any files.")

In [ ]:
# === UNCOMMENT AND EDIT if optimizer found better params ===
# Replace values with the best config from the optimizer output above.
# Then run this cell to patch settings.py.

# NEW_ALPHA = 16        # Replace with optimizer's best alpha
# NEW_W_TI = 0.90       # Replace with optimizer's best w_ti
# NEW_W_3D = 0.45       # Replace with optimizer's best w_3d  
# NEW_GAMMA = 0.10      # Replace with optimizer's best gamma
# NEW_CAL_MODE = "gram" # Replace with optimizer's best cal_mode

# settings_path = PROJECT_ROOT / "src" / "config" / "settings.py"
# text = settings_path.read_text()
# import re
# text = re.sub(r'CMSCI_MARGIN_ALPHA = \d+', f'CMSCI_MARGIN_ALPHA = {NEW_ALPHA}', text)
# text = re.sub(r'CMSCI_CHANNEL_WEIGHT_TI = [\d.]+', f'CMSCI_CHANNEL_WEIGHT_TI = {NEW_W_TI}', text)
# text = re.sub(r'CMSCI_W_3D = [\d.]+', f'CMSCI_W_3D = {NEW_W_3D}', text)
# text = re.sub(r'CMSCI_GAMMA = [\d.]+', f'CMSCI_GAMMA = {NEW_GAMMA}', text)
# text = re.sub(r'CMSCI_CALIBRATION_MODE = "\w+"', f'CMSCI_CALIBRATION_MODE = "{NEW_CAL_MODE}"', text)
# settings_path.write_text(text)
# print("settings.py updated! Restart kernel or re-import to pick up changes.")

---
## Task 4: Scale Benchmark Evaluation to 1000 Samples

Now that we have 1000 AudioCaps samples downloaded (from Task 1), run the full benchmark evaluation with all methods.

In [ ]:
print("=" * 60)
print("TASK 4: Benchmark Evaluation (1000 AudioCaps samples)")
print("=" * 60)

t0 = time.time()
!python scripts/evaluate_benchmarks.py --audiocaps --max-samples 1000
print(f"\nBenchmark evaluation completed in {time.time()-t0:.0f}s")

# Display results
results_path = PROJECT_ROOT / "runs" / "benchmarks" / "benchmark_evaluation.json"
if results_path.exists():
    with open(results_path) as f:
        results = json.load(f)
    print("\n" + "=" * 60)
    print("BENCHMARK RESULTS SUMMARY")
    print("=" * 60)
    for ds_name, ds_data in results.items():
        print(f"\n  Dataset: {ds_name} ({ds_data.get('n_entries', '?')} entries)")
        methods = ds_data.get('methods', {})
        print(f"  {'Method':<22} {'AUC':>6} {'Acc':>6}")
        print(f"  {'-'*36}")
        for name, m in sorted(methods.items(), key=lambda x: -x[1].get('auc', 0)):
            print(f"  {name:<22} {m.get('auc', 0):>6.4f} {m.get('accuracy', 0):>6.4f}")

---
## Task 5: Re-run Full Evaluation + Robustness

After retraining models and re-optimizing hyperparameters, re-run the full evaluation pipeline to get updated numbers for the paper.

In [ ]:
print("=" * 60)
print("TASK 5a: Full Evaluation (all baselines vs human ratings)")
print("=" * 60)

t0 = time.time()
!python scripts/run_full_evaluation.py --all-samples --skip-vlm --skip-blip
print(f"\nFull evaluation completed in {time.time()-t0:.0f}s")

In [ ]:
print("=" * 60)
print("TASK 5b: Sensitivity Analysis")
print("=" * 60)

t0 = time.time()
!python scripts/run_sensitivity.py
print(f"\nSensitivity analysis completed in {time.time()-t0:.0f}s")

In [ ]:
print("=" * 60)
print("TASK 5c: Seed Robustness (10 seeds)")
print("=" * 60)

t0 = time.time()
!python scripts/run_seed_robustness.py
print(f"\nSeed robustness completed in {time.time()-t0:.0f}s")

---
## Task 6 (Optional): VLM-as-Judge

Requires Ollama running with a vision-language model. Skip if not available.

```bash
# Install Ollama first (if on a machine with Ollama access):
# curl -fsSL https://ollama.com/install.sh | sh
# ollama pull llava:7b
```

In [ ]:
# Check if Ollama is available
import urllib.request

ollama_available = False
try:
    resp = urllib.request.urlopen("http://localhost:11434/api/tags", timeout=5)
    models = json.loads(resp.read())
    model_names = [m["name"] for m in models.get("models", [])]
    print(f"Ollama available! Models: {model_names}")
    ollama_available = True
except Exception as e:
    print(f"Ollama not available ({e})")
    print("Skipping VLM-as-Judge. This is optional for the paper.")

In [ ]:
if ollama_available:
    print("=" * 60)
    print("TASK 6: VLM-as-Judge")
    print("=" * 60)
    t0 = time.time()
    !python scripts/run_full_evaluation.py --all-samples --skip-blip
    print(f"\nVLM evaluation completed in {time.time()-t0:.0f}s")
else:
    print("Skipped — Ollama not running.")

---
## Summary: Collect All Results

In [ ]:
print("=" * 60)
print("ALL GPU TASKS COMPLETE — RESULTS SUMMARY")
print("=" * 60)

# Check all result files
result_files = {
    "Full Evaluation": "runs/full_evaluation/full_evaluation.json",
    "Sensitivity": "runs/sensitivity/sensitivity_analysis.json",
    "Seed Robustness": "runs/robustness/seed_robustness.json",
    "NegBank Robustness": "runs/robustness/negbank_robustness.json",
    "Benchmark": "runs/benchmarks/benchmark_evaluation.json",
    "CLAP Sensitivity": "runs/clap_sensitivity/clap_sensitivity.json",
    "Failure Analysis": "runs/failure_analysis/failure_analysis.json",
}

print("\n  Result files:")
for name, path in result_files.items():
    full_path = PROJECT_ROOT / path
    if full_path.exists():
        size = full_path.stat().st_size / 1024
        print(f"    {name:25s} — {path} ({size:.1f} KB)")
    else:
        print(f"    {name:25s} — MISSING")

# Show key metrics
print("\n" + "-" * 60)
print("KEY METRICS:")
print("-" * 60)

eval_path = PROJECT_ROOT / "runs" / "full_evaluation" / "full_evaluation.json"
if eval_path.exists():
    with open(eval_path) as f:
        eval_data = json.load(f)
    results = eval_data.get("results", {})
    print(f"\n  {'Method':25s} {'rho':>7s} {'p-value':>10s} {'Sig':>4s}")
    print(f"  {'-'*50}")
    for method, corr in sorted(results.items(), key=lambda x: -x[1].get('rho', -999)):
        rho = corr.get('rho', float('nan'))
        p = corr.get('p', float('nan'))
        sig = '*' if p < 0.05 else ''
        print(f"  {method:25s} {rho:7.4f} {p:10.6f} {sig:>4s}")

bench_path = PROJECT_ROOT / "runs" / "benchmarks" / "benchmark_evaluation.json"
if bench_path.exists():
    with open(bench_path) as f:
        bench_data = json.load(f)
    for ds_name, ds_result in bench_data.items():
        print(f"\n  Benchmark ({ds_name}, n={ds_result.get('n_entries', '?')}):")
        for method, m in sorted(ds_result.get('methods', {}).items(), key=lambda x: -x[1].get('auc', 0)):
            print(f"    {method:22s} AUC={m.get('auc', 0):.4f}")

print("\n" + "=" * 60)
print("DOWNLOAD THE FOLLOWING TO YOUR LAPTOP:")
print("=" * 60)
print("""
  1. models/bridge/bridge_best.pt + .json
  2. models/prob_adapters/clip_adapter.pt + .json  
  3. models/prob_adapters/clap_adapter.pt + .json
  4. runs/  (entire directory — all results)
  5. src/config/settings.py (if you updated hyperparams)
""")

In [ ]:
# Create a ZIP of all results + models for easy download
import shutil
import zipfile

zip_path = PROJECT_ROOT / "gpu_results.zip"
print("Creating results package...")

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    # Models
    for model_file in (PROJECT_ROOT / "models").rglob("*"):
        if model_file.is_file() and model_file.suffix in (".pt", ".json"):
            arcname = str(model_file.relative_to(PROJECT_ROOT))
            zf.write(model_file, arcname)
            
    # Results
    for run_file in (PROJECT_ROOT / "runs").rglob("*.json"):
        arcname = str(run_file.relative_to(PROJECT_ROOT))
        zf.write(run_file, arcname)
    
    # Figures
    for fig_file in (PROJECT_ROOT / "runs").rglob("*.pdf"):
        arcname = str(fig_file.relative_to(PROJECT_ROOT))
        zf.write(fig_file, arcname)
    
    # Settings (in case updated)
    settings_file = PROJECT_ROOT / "src" / "config" / "settings.py"
    if settings_file.exists():
        zf.write(settings_file, "src/config/settings.py")
    
    # Calibration
    cal_file = PROJECT_ROOT / "artifacts" / "cmsci_calibration.json"
    if cal_file.exists():
        zf.write(cal_file, "artifacts/cmsci_calibration.json")

zip_size = zip_path.stat().st_size / (1024 * 1024)
print(f"\nPackage created: {zip_path} ({zip_size:.1f} MB)")
print("Download this ZIP and extract into your project directory on your laptop.")